# Clase 11 · Notebook 01
# Métricas avanzadas: umbral, ROC/AUC y datos desbalanceados

**Introducción al Deep Learning — Módulo 2, Unidad 2: Clasificación (Parte II)**

Profundizamos en la **evaluación** de un clasificador binario:

1. El **umbral** de decisión y su efecto en precision/recall.
2. La curva **ROC** y el **AUC**.
3. La curva **Precision-Recall**.
4. **Datos desbalanceados** y el uso de `class_weight`.

Usamos el dataset de cáncer de mama.

> 💡 En Colab, TensorFlow ya viene instalado. Ejecuta las celdas en orden con **Shift + Enter**.

## 1. Entrenar un clasificador binario

In [ ]:
try:
    import tensorflow as tf
except ImportError:
    import sys, subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tensorflow"])
    import tensorflow as tf
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
tf.random.set_seed(42); np.random.seed(42)

data = load_breast_cancer()
X = StandardScaler().fit_transform(data.data)
y = data.target            # 1 = benigno, 0 = maligno
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
modelo = Sequential([Input(shape=(X.shape[1],)), Dense(16, activation="relu"), Dense(1, activation="sigmoid")])
modelo.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
modelo.fit(Xtr, ytr, epochs=40, batch_size=16, verbose=0)
probs = modelo.predict(Xte, verbose=0).ravel()
print("Modelo entrenado. Probabilidades de ejemplo:", probs[:5].round(3))

## 2. El umbral de decisión

El modelo da **probabilidades**. Para decidir la clase aplicamos un umbral. Veamos cómo cambian precision
y recall al moverlo.

In [ ]:
from sklearn.metrics import precision_score, recall_score

for u in [0.3, 0.5, 0.7]:
    pred = (probs >= u).astype(int)
    pr = precision_score(yte, pred)
    rc = recall_score(yte, pred)
    print(f"umbral {u}:  precision = {pr:.3f}  |  recall = {rc:.3f}")

Al **bajar** el umbral detectamos más positivos (sube el recall) a costa de la precision, y al revés.
El umbral se elige según el coste de cada error.

## 3. Curva ROC y AUC

La curva ROC recorre todos los umbrales (tasa de verdaderos positivos vs. falsos positivos).
El **AUC** resume su calidad en un número (1 = perfecto, 0,5 = azar).

In [ ]:
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

fpr, tpr, _ = roc_curve(yte, probs)
auc_val = auc(fpr, tpr)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, color="#FF647E", lw=2, label=f"ROC (AUC = {auc_val:.3f})")
plt.plot([0, 1], [0, 1], "--", color="gray", label="azar")
plt.xlabel("Tasa de falsos positivos (FPR)"); plt.ylabel("Tasa de verdaderos positivos (TPR)")
plt.title("Curva ROC"); plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()
print("AUC:", round(auc_val, 3))

## 4. Curva Precision-Recall

Más informativa cuando la clase positiva es rara.

In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score

pr, rc, _ = precision_recall_curve(yte, probs)
ap = average_precision_score(yte, probs)
plt.figure(figsize=(6, 5))
plt.plot(rc, pr, color="#000F9F", lw=2)
plt.xlabel("Recall"); plt.ylabel("Precision")
plt.title(f"Curva Precision-Recall (AP = {ap:.3f})")
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 5. Datos desbalanceados y `class_weight`

Creamos una versión **desbalanceada** (pocas muestras de la clase maligna) y comparamos el recall de esa
clase con y sin `class_weight`.

In [ ]:
# Construir un conjunto desbalanceado: pocas muestras de la clase 0 (maligno)
idx0 = np.where(ytr == 0)[0][:15]      # solo 15 malignos
idx1 = np.where(ytr == 1)[0]
sel = np.concatenate([idx0, idx1]); np.random.shuffle(sel)
Xtr_d, ytr_d = Xtr[sel], ytr[sel]
print("Distribución entrenamiento desbalanceado:", np.bincount(ytr_d), "(maligno, benigno)")

def entrenar(class_weight=None):
    m = Sequential([Input(shape=(X.shape[1],)), Dense(16, activation="relu"), Dense(1, activation="sigmoid")])
    m.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    m.fit(Xtr_d, ytr_d, epochs=40, batch_size=16, verbose=0, class_weight=class_weight)
    pred = (m.predict(Xte, verbose=0).ravel() >= 0.5).astype(int)
    return recall_score(yte, pred, pos_label=0)   # recall de la clase MALIGNA (la rara)

r_sin = entrenar(None)
r_con = entrenar({0: 6.0, 1: 1.0})    # damos más peso a la clase rara
print(f"\nRecall de la clase maligna (la importante):")
print(f"  sin class_weight : {r_sin:.3f}")
print(f"  con class_weight : {r_con:.3f}")

Dar más peso a la clase minoritaria mejora su **recall**: el modelo deja de ignorarla.

## Resumen

- El **umbral** equilibra precision y recall; ajústalo según el coste de cada error.
- La **curva ROC/AUC** y la **Precision-Recall** evalúan el modelo en todos los umbrales.
- Con **datos desbalanceados**, la accuracy engaña; `class_weight` (o re-muestreo) ayuda a la clase rara.

➡️ En el **Notebook 02** optimizaremos los hiperparámetros del modelo con búsqueda aleatoria.